# DeiT-Tiny seed5 — E→O raw top1 vs Vanilla + rollback

Attach CIFAR-100; enable Internet, T4x2 and Kaggle Secret `github_token`.
Optional resume: full output archive from this same seed/recipe. Compatible
seed5 E→O progress from the earlier raw-vs-Bypass notebook can be reused only
with its original hash-bound fork/raw reference and matching configuration.
Each requested arm resumes independently. No Bypass, O-only, A3/A4 or extra arm.

Phase1 trains Vanilla for seed5 until150 epochs without a strict validation
accuracy best. No minimum epoch and no requirement to finish schedule300.
Reload that historical-best model, AdamW, scheduler and RNG/loader state.

Phase2 runs two independent150-epoch trajectories from identical full fork:
- GPU0: `e_driven_o_raw`, original TINY top1 WHERE over12 MLPs, one E→O projection.
- GPU1: `vanilla_rollback`, genuine Vanilla training without E/projection/moment reset.

Both use patience10. Higher accuracy OR an exact accuracy tie with lower loss
updates the controller anchor and resets stall; otherwise stall increments once.
Rollback restores model/optimizer/scheduler and retains the current RNG/loader
stream. No retrigger. Report best uses strict accuracy and paired loss.
Scientific escape compares observed post-fork best to this seed's historical best.
Vanilla rollback is newly trained, not the observed Vanilla plateau-confirmation
window. On resume completed arms skip training and remaining arms use free GPU slots.

Recipe: batch64, AdamW LR5e-4/WD.05, warmup5 + cosine300, plateau150, horizon150,
max_epoch800. E→O rank4, stats256, WHERE3x32, projection32, gate32, CG200,
scales(.0125,.025,.05). Evaluation uses3000 validation samples;2000 reserved
samples do not trigger/select anything. Official test is unused.

Every epoch saves full state including anchor/counter, optimizer, scheduler,
RNG and loader. Download the full output tarball for another session.
The comparison uses completed current arms with identical fork/protocol/horizon;
failed or unrequested results do not enter the accuracy delta.


In [ ]:
import json, os, shutil, subprocess, sys
from pathlib import Path
from kaggle_secrets import UserSecretsClient

REPO = Path("/kaggle/working/deit-all-arms-repo")
OUTPUT = Path("/kaggle/working/projection_log_analysis")
URL = "https://github.com/duyh80456-code/Counterfactual-Projection.git"
token = UserSecretsClient().get_secret("github_token").strip()
if not token:
    raise RuntimeError("Enable the github_token Kaggle Secret")
askpass = Path("/kaggle/working/.projection_log_git_askpass.py")
askpass.write_text("#!/usr/bin/env python3\nimport os,sys\np=sys.argv[1] if len(sys.argv)>1 else ''\nprint('x-access-token' if 'Username' in p else os.environ['GITHUB_TOKEN_RUNTIME'])\n")
askpass.chmod(0o700)
env = os.environ.copy()
env.update(GITHUB_TOKEN_RUNTIME=token, GIT_ASKPASS=str(askpass), GIT_TERMINAL_PROMPT="0")
try:
    if not REPO.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "main", URL, str(REPO)], env=env, check=True)
    else:
        subprocess.run(["git", "-C", str(REPO), "fetch", "--depth", "1", "origin", "main"], env=env, check=True)
        subprocess.run(["git", "-C", str(REPO), "merge", "--ff-only", "origin/main"], env=env, check=True)
finally:
    askpass.unlink(missing_ok=True)
    env.pop("GITHUB_TOKEN_RUNTIME", None)
    del token
GROMO = Path("/kaggle/working/deit-gromo")
GROMO_COMMIT = "8d19107b61a9459a9021065a329b699adcb0f25b"
if not GROMO.exists():
    subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout",
                    "https://github.com/growingnet/gromo.git", str(GROMO)], check=True)
subprocess.run(["git", "-C", str(GROMO), "fetch", "--depth", "1", "origin", GROMO_COMMIT], check=True)
subprocess.run(["git", "-C", str(GROMO), "checkout", "--detach", GROMO_COMMIT], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO}[vision,test]", "-e", str(GROMO)], check=True)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
ENV = os.environ.copy()
ENV["PYTHONPATH"] = os.pathsep.join((str(REPO), str(GROMO / "src")))
ENV["REQUIRE_DEIT_INTEGRATION"] = "1"
subprocess.run([sys.executable, "-m", "pytest", "-q",
    "tests/test_deit_function_preserving.py", "tests/test_deit_sites.py",
    "tests/test_deit_projection.py", "tests/test_deit_no_aux_persistence.py",
    "tests/test_deit_protocol.py", "tests/test_deit_vanilla_reference.py", "tests/test_deit_e_rollback.py", "tests/test_deit_ablation.py", "tests/test_deit_ablation_integration.py", "tests/test_deit_logging.py", "tests/test_deit_parallel.py", "tests/test_deit_resume.py", "tests/test_kaggle_checkpoint_discovery.py"], cwd=REPO, env=ENV, check=True)
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Enable a CUDA GPU; this notebook trains DeiT")
print("Repository commit:", subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip())


In [ ]:
from dataclasses import asdict
from experiments.deit_protocol import DeitRecipe, protocol, canonical_model_config
from adapters.deit_ablation import res18_cp_config
from experiments.shared_protocol import sha256_file

SEED = 5
ARMS = ('e_driven_o_raw', 'vanilla_rollback')
SCHEDULE_EPOCHS = 300
STALL_PATIENCE = 150
POST_FORK_EPOCHS = 150
ALGORITHM_PATIENCE = 10
MAX_EPOCH = 800
BATCH_SIZE = 64
GPU_DEVICES = 'auto'  # T4x2: two workers; one GPU: one worker.
LEARNING_RATE = 5e-4  # Explicit per-arm LR; no implicit ImageNet batch scaling.
RANK = 4
PROJECTION_SAMPLES = 32
SCALES = '.0125,.025,.05'
OUTPUT = Path(f'/kaggle/working/deit_tiny_seed{SEED}_raw_vs_vanilla_rollback_v1')
OUTPUT.mkdir(parents=True, exist_ok=True)
RECIPE = DeitRecipe(seed=SEED, batch_size=BATCH_SIZE, learning_rate=LEARNING_RATE,
    schedule_epochs=SCHEDULE_EPOCHS, stall_start_epoch=0,
    stall_patience=STALL_PATIENCE, reference_epochs=POST_FORK_EPOCHS, max_epoch=MAX_EPOCH)
RECIPE.validate()
EXPECTED = protocol(RECIPE, canonical_model_config())
CP = res18_cp_config()
from dataclasses import replace
CP = replace(CP, rank=RANK, projection_samples=PROJECTION_SAMPLES,
             scales=tuple(map(float, SCALES.split(','))))
roots = set()
for directory, _, files in os.walk('/kaggle/input', followlinks=True):
    if Path(directory).name == 'cifar-100-python' and {'train', 'meta'}.issubset(files):
        roots.add(Path(directory).parent.resolve())
if len(roots) != 1:
    raise FileNotFoundError(f'Attach exactly one CIFAR-100 dataset: {roots}')
DATA_ROOT = next(iter(roots))
print('Recipe:', asdict(RECIPE))
print('Projection:', asdict(CP))


In [ ]:
# Import full outputs (original .pt, .tar.gz/.zip, or mounted notebook output).
import shutil
from experiments.deit_resume import prepare_resume
RESUME_PLAN = prepare_resume(['/kaggle/input'], OUTPUT, RECIPE, CP,
    horizon=POST_FORK_EPOCHS, patience=ALGORITHM_PATIENCE, inner_steps=5, arms=ARMS)
PHASE1 = OUTPUT / 'vanilla_stall'
FORK = PHASE1 / 'plateau_checkpoint.pt'
VANILLA_REFERENCE = PHASE1 / 'vanilla_reference.pt'


In [ ]:
command = [sys.executable, '-m', 'experiments.run_deit_all_arms', '--data-root', str(DATA_ROOT),
    '--output', str(OUTPUT), '--seed', str(SEED), '--schedule-epochs', str(SCHEDULE_EPOCHS),
    '--max-epoch', str(MAX_EPOCH), '--batch-size', str(BATCH_SIZE), '--learning-rate', str(LEARNING_RATE),
    '--stall-patience', str(STALL_PATIENCE), '--post-fork-epochs', str(POST_FORK_EPOCHS),
    '--algorithm-patience', str(ALGORITHM_PATIENCE), '--rank', str(RANK),
    '--projection-samples', str(PROJECTION_SAMPLES), '--scales', SCALES, '--opt-inner-steps', '5',
    '--gpu-devices', GPU_DEVICES, '--arms', ','.join(ARMS)]
if FORK.exists():
    command += ['--plateau-checkpoint', str(FORK), '--vanilla-reference', str(VANILLA_REFERENCE)]
try:
    subprocess.run(command, cwd=REPO, env=ENV, check=True)
finally:
    archive = shutil.make_archive(str(OUTPUT), 'gztar', root_dir=OUTPUT)
    print('Resumable output archive:', archive)


In [ ]:
from experiments.shared_protocol import atomic_json_save
status_path = OUTPUT / 'arm_status.json'
statuses = json.loads(status_path.read_text()) if status_path.exists() else {}
comparison = {}
expected_hash = sha256_file(FORK)
for method in ARMS:
    status = statuses.get(method, {})
    if status.get('status') != 'completed':
        comparison[method] = {'available': False, 'status': status.get('status', 'not_completed')}
        continue
    result = json.loads((OUTPUT / 'arms' / method / 'result.json').read_text())
    identity = result['run_identity']
    if (result['theta_best_hash'] != expected_hash or result['protocol'] != EXPECTED
            or identity['cp_config'] != json.loads(json.dumps(asdict(CP)))
            or identity['algorithm_patience'] != ALGORITHM_PATIENCE
            or identity['post_fork_epochs'] != POST_FORK_EPOCHS
            or result['history'][-1]['post_fork_epoch'] != POST_FORK_EPOCHS):
        raise ValueError('Seed5 comparison requires the same complete fork/protocol/config/horizon')
    comparison[method] = {'available': True, **{key: result[key] for key in (
        'fork_epoch', 'historical_best_accuracy', 'report_best_accuracy',
        'report_best_loss', 'report_best_epoch', 'delta_vs_historical_best', 'scientific_escape')},
        'rollback_count': len(result['rollback_events']), 'fork_hash': expected_hash}
raw = comparison.get('e_driven_o_raw', {})
vanilla = comparison.get('vanilla_rollback', {})
eligible = raw.get('available', False) and vanilla.get('available', False)
report = {'arms': comparison, 'comparison_eligible': eligible,
    'e2o_minus_vanilla_rollback_best_accuracy':
        raw['report_best_accuracy'] - vanilla['report_best_accuracy'] if eligible else None}
atomic_json_save(report, OUTPUT / 'raw_vs_vanilla_rollback_comparison.json')
print(json.dumps(report, indent=2))
print('Status:', statuses)
